# 🦙 Ollama Notebook

A hands-on introduction to running local LLMs with **Ollama** and its Python client.

| Section | Topic |
|---------|-------|
| 1 | Setup & Installation |
| 2 | Simple Stateless Call |
| 3 | Memory / Stateful Conversation |
| 4 | Text Embeddings & Bag of Words |
| 5 | Tool Calling: `sum_two_numbers` |

> **Prerequisites:** Ollama must be running locally (`ollama serve`) and you must have
> pulled `llama3.2` and `nomic-embed-text` (`ollama pull nomic-embed-text`).

---
## Section 1 — Setup

Install the Ollama Python client and verify the connection to your local server.

In [ ]:
# Install the Ollama Python client (run once)
# Uncomment the line below if you haven't installed it yet
# !pip install ollama
#qwen2:1.5b uses this instead 
import ollama

# Quick sanity check — list models available on the local server
models = ollama.list()

print('Available models:')
for m in models.get('models', []):
    print(f"  - {m['model']}  ({round(m.get('size', 0) / 1e9, 1)} GB)")
print()
print('✅ Ollama is running and accessible!')

---
## Section 2 — Simple Stateless Call

Each call is **completely independent** — the model has no memory of previous messages.

Use this for single-turn, one-off questions.

```
You  ──► [model]  ──► Reply
     (no history)
```

In [ ]:
# from pyexpat import model
# A single, stateless question
response = ollama.chat(
    # model='llama3.2',
    model = 'qwen3.5',
    messages=[{'role': 'user', 'content': 'What is the capital of France?'}]
)

print('Response:', response['message']['content'])

In [ ]:
# Inspect the full response object
print('Model  :', response['model'])
print('Done   :', response['done'])
print('Reason :', response.get('done_reason', 'n/a'))

# Token usage (if available)
print('Prompt tokens    :', response.get('prompt_eval_count', 'n/a'))
print('Completion tokens:', response.get('eval_count', 'n/a'))

### 🔍 Key insight

Notice that `messages` contains **only one message** here. The model answers without
any conversational context — if you ask *"What is my name?"* in the next call,
it won't know because you never told it in *this* call.

---
## Section 3 — Memory / Stateful Conversation

LLMs are stateless by design — they don't remember between API calls.
To simulate memory, we **accumulate all messages** (user + assistant) in a list
and send the full history with every request.

```
Turn 1:  [user: 'My name is Mohamed']                      ──► model
Turn 2:  [user: '...', asst: '...', user: 'My name?']      ──► model
Turn 3:  [user: '...', asst: '...', ..., user: 'How old?'] ──► model
```

In [ ]:
# messages list accumulates the entire conversation history
messages = []

def chat_with_memory(user_input: str) -> str:
    """Send a message and maintain full conversation history."""
    # 1. Append the new user message
    messages.append({'role': 'user', 'content': user_input})

    # 2. Send the FULL history to the model
    response = ollama.chat(model='qwen2:1.5b', messages=messages)
    reply = response['message']['content']

    # 3. Append the assistant's reply so it's included next time
    messages.append({'role': 'assistant', 'content': reply})
    return reply

In [ ]:
# Turn 1 — introduce ourselves
reply1 = chat_with_memory('My name is Mohamed and I love Python.')
print('Turn 1:', reply1)
print()

In [ ]:
# Turn 2 — test if the model remembers
reply2 = chat_with_memory('What is my name?')
print('Turn 2:', reply2)
print()

In [ ]:
# Turn 3 — continue the thread
reply3 = chat_with_memory('What programming language did I say I love?')
print('Turn 3:', reply3)
print()

# Show the full message history that was built up
print(f'Total messages in history: {len(messages)}')
for i, msg in enumerate(messages, 1):
    role = msg['role'].upper().ljust(9)
    preview = msg['content'][:80].replace('\n', ' ')
    print(f'  [{i}] {role}: {preview}...' if len(msg['content']) > 80 else f'  [{i}] {role}: {preview}')

### ⚠️ Memory Trade-Off

| Pro | Con |
|-----|-----|
| Model remembers the full conversation | Each call sends more tokens |
| Simple to implement | Costs grow with conversation length |
| Works with any Ollama model | Very long chats may hit the context window limit |

> **Tip:** For production apps, trim or summarise old messages to keep the history manageable.

---
## Section 4 — Text Embeddings & Bag of Words (BoW)

In natural language processing, we must convert textual data into numbers before passing them to machine learning models. Two common methods are:
1. **Bag of Words (BoW)**: A simple count-based representation.
2. **Text Embeddings**: Dense vector representations that capture semantic meaning.

### 1. Bag of Words (BoW) Example
Bag of Words counts how many times each word appears in the text. It ignores grammar, order, and semantic relations.
Here is a quick Python implementation demonstrating BoW:

In [ ]:
# Simple Bag of Words example
import numpy as np

# A small vocabulary of unique words
vocab = {"i": 0, "love": 1, "python": 2, "coding": 3}
print("Vocabulary:", vocab)

def get_bow_vector(sentence: str) -> list:
    # Initialize count vector
    vector = [0] * len(vocab)
    words = sentence.lower().split()
    for word in words:
        if word in vocab:
            vector[vocab[word]] += 1
    return vector

sent1 = "I love Python"
sent2 = "I love coding"

vec1 = get_bow_vector(sent1)
vec2 = get_bow_vector(sent2)

print(f"'{sent1}' -> Vector: {vec1}")
print(f"'{sent2}' -> Vector: {vec2}")

# Compute similarity using dot product
dot_product = np.dot(vec1, vec2)
print("Dot Product Similarity:", dot_product)

### 2. Text Embeddings with Ollama
Embeddings represent text in a continuous vector space where words/sentences with similar meaning are close to each other.

For example, "Python" and "coding" have high semantic similarity, which Bag of Words cannot capture because they are completely different vocabulary tokens.

Let's generate a 768-dimensional text embedding locally using the `nomic-embed-text` model with Ollama:

In [ ]:
import ollama

# Generate embeddings for a text prompt
response = ollama.embeddings(
    model='nomic-embed-text',
    prompt='Llamas are great animals!'
)

# Extract and inspect the vector
embedding = response.get('embedding', [])
print("Embedding successfully generated!")
print("Vector length:", len(embedding))
print("First 5 values:", embedding[:5])

---
## Section 5 — Tool Calling: `sum_two_numbers`

**Tools** (function calling) let the LLM trigger real Python code instead of guessing.
The model decides *when* to call a tool and *what arguments* to pass.

### The Tool Calling Loop

```
1. You send prompt + tool schemas
        │
        ▼
2. Model returns tool_call  ◄── model does NOT answer yet
        │
        ▼
3. Your code executes the function
        │
        ▼
4. You send the result back to the model
        │
        ▼
5. Model gives a final natural-language reply
```

In [ ]:
import json

# ── Step 1: Define the tool schema ──────────────────────────────────────
# This JSON object tells the model: 'You can call this function'
tools = [{
    'type': 'function',
    'function': {
        'name': 'sum_two_numbers',
        'description': 'Returns the sum of two numbers. Use this whenever the user asks to add numbers.',
        'parameters': {
            'type': 'object',
            'properties': {
                'a': {'type': 'number', 'description': 'The first number'},
                'b': {'type': 'number', 'description': 'The second number'}
            },
            'required': ['a', 'b']
        }
    }
}]

print('Tool schema defined:')
print(json.dumps(tools[0]['function'], indent=2))

In [ ]:
# ── Step 2: Implement the actual Python function ─────────────────────────
def sum_two_numbers(a: float, b: float) -> float:
    """The real implementation the LLM will trigger."""
    return a + b

# Quick test
print('Direct function test: sum_two_numbers(7, 15) =', sum_two_numbers(7, 15))

In [ ]:
# ── Step 3: Send the request with tools attached ─────────────────────────
user_question = 'What is 7 plus 15?'

response = ollama.chat(
    model='qwen3.5',
    messages=[{'role': 'user', 'content': user_question}],
    tools=tools
)

message = response['message']
print('Model raw response:')
print('  Content     :', message.get('content', '(none)'))
print('  Tool calls  :', message.get('tool_calls', '(none)'))

In [ ]:
# ── Step 4 & 5: Execute the tool and send the result back ────────────────
if message.get('tool_calls'):
    for tool_call in message['tool_calls']:
        fn_name = tool_call['function']['name']
        args    = tool_call['function']['arguments']

        print(f'Model requested tool : {fn_name}')
        print(f'With arguments       : {args}')

        # Execute the corresponding Python function
        if fn_name == 'sum_two_numbers':
            result = sum_two_numbers(**args)
            print(f'Tool result          : {result}')
            print()

            # Send the tool result back to the model so it can answer
            final_response = ollama.chat(
                model='qwen3.5',
                messages=[
                    {'role': 'user',      'content': user_question},
                    {'role': 'assistant', 'content': message.get('content', ''),
                     'tool_calls': message['tool_calls']},
                    {'role': 'tool',      'content': str(result)}
                ]
            )

            print('Final model answer:', final_response['message']['content'])
else:
    # Model answered directly without using a tool (fallback)
    print('Model answered directly (no tool call):')
    print(message['content'])

### 🔁 Wrapping Up: Full Tool Calling Function

Here is the complete, reusable version of the tool-calling pattern above:

In [ ]:
def run_with_tools(user_input: str, tools: list, tool_map: dict) -> str:
    """
    Generic tool-calling loop.

    Args:
        user_input : The user's question.
        tools      : List of tool schema dicts (sent to the model).
        tool_map   : Dict mapping function name -> Python callable.

    Returns:
        The model's final natural-language answer.
    """
    messages = [{'role': 'user', 'content': user_input}]

    # Step 1: Initial request with tools
    response = ollama.chat(model='qwen3.5', messages=messages, tools=tools)
    msg = response['message']

    # Step 2: Process all tool calls (there may be more than one)
    if msg.get('tool_calls'):
        messages.append({'role': 'assistant', 'content': msg.get('content', ''),
                         'tool_calls': msg['tool_calls']})

        for tc in msg['tool_calls']:
            fn_name = tc['function']['name']
            args    = tc['function']['arguments']
            if fn_name in tool_map:
                result = tool_map[fn_name](**args)     # execute
                messages.append({'role': 'tool', 'content': str(result)})

        # Step 3: Get final answer from model
        final = ollama.chat(model='qwen3.5', messages=messages)
        return final['message']['content']

    return msg.get('content', '')


# ── Test it ──────────────────────────────────────────────────────────────
tool_map = {'sum_two_numbers': sum_two_numbers}

print(run_with_tools('Add 42 and 58 for me.', tools, tool_map))
print(run_with_tools('What is 100 plus 200?', tools, tool_map))

---
## 🎉 Summary

| Concept | Key Takeaway |
|---------|-------------|
| **Stateless call** | One message → one reply, no history |
| **Stateful call** | Pass the full `messages` list every time to simulate memory |
| **Tool calling** | Model requests a function → you execute it → send result back |

**Next steps:**
- Try a different model: change `'llama3.2'` to `'mistral'` or `'gemma2'`
- Add more tools to `tool_map` (e.g., a weather API, a calculator, a database query)
- Add a system prompt: `{'role': 'system', 'content': 'You are a helpful assistant.'}` at the start of `messages`